In [28]:
from pathlib import Path
from os import path as osp
import numpy as np
from av2.datasets.sensor.av2_sensor_dataloader import AV2SensorDataLoader
from av2.map.map_api import ArgoverseStaticMap
from shapely.geometry import Polygon, MultiPolygon
from copy import deepcopy
from av2_to_wgs_conversion import convert_city_coords_to_wgs84
import requests

In [30]:
overpass_useragent = 'NeurIPS' # use a more unique useragent here for yourself
overpass_headers = {
    'Connection': 'keep-alive',
    'sec-ch-ua': '"Google Chrome 80"',
    'Accept': '*/*',
    'Sec-Fetch-Dest': 'empty',
    'User-Agent': overpass_useragent,
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'Origin': 'https://overpass-turbo.eu',
    'Sec-Fetch-Site': 'cross-site',
    'Sec-Fetch-Mode': 'cors',
    'Referer': 'https://overpass-turbo.eu/',
    'Accept-Language': '',
    'dnt': '1',
}

FAIL_LOGS = [
    # official
    '75e8adad-50a6-3245-8726-5e612db3d165',
    '54bc6dbc-ebfb-3fba-b5b3-57f88b4b79ca',
    'af170aac-8465-3d7b-82c5-64147e94af7d',
    '6e106cf8-f6dd-38f6-89c8-9be7a71e7275',
    # observed
    '01bb304d-7bd8-35f8-bbef-7086b688e35e',
    '453e5558-6363-38e3-bf9b-42b5ba0a6f1d',
    # observed ll2_custom
    '8940f5f1-13e0-3094-99ba-da2d17639774', 
    'c08279c0-10b4-3d21-b13f-a1c1a0b87f8b', 
    'c96a09c8-46ed-391f-8a66-c46fa8b76029'
]

av2_root_path = "/tmp/argoverse2/sensor"
av2_geo_split_root_path = "geographical-splits/near_extrapolation_splits/argoverse2/argo_geo_txts"
splits = ['train', 'val', 'test']

out_path = "out"

def get_log_ids(geo_split_root_path, split):
    log_ids_file_path = osp.join(geo_split_root_path, split + '.txt')
    log_ids = []
    with open(log_ids_file_path, 'r') as f:
        for line in f:
            log_ids.append(line.strip())
    for l in FAIL_LOGS:
        if l in log_ids:
            log_ids.remove(l)
    return log_ids

already_processed_scenario_maps = {}

In [54]:

log_ids_train = get_log_ids(av2_geo_split_root_path, 'train')
log_ids_val = get_log_ids(av2_geo_split_root_path, 'val')
log_ids_test = get_log_ids(av2_geo_split_root_path, 'test')

root_path_train = osp.join(av2_root_path, 'train')
root_path_val = osp.join(av2_root_path, 'val')
root_path_test = osp.join(av2_root_path, 'test')

loader_train = AV2SensorDataLoader(Path(root_path_train), Path(root_path_train))
loader_val = AV2SensorDataLoader(Path(root_path_val), Path(root_path_val))
loader_test = AV2SensorDataLoader(Path(root_path_test), Path(root_path_test))

log_ids_train_og = list(loader_train.get_log_ids())
log_ids_val_og = list(loader_val.get_log_ids())
log_ids_test_og = list(loader_test.get_log_ids())
# Initialize the dictionaries
loader = {}
root_path = {}

for log_id in log_ids_train:
            if log_id in log_ids_train_og:
                loader[log_id] = loader_train
                root_path[log_id] = root_path_train
            elif log_id in log_ids_val_og:
                loader[log_id] = loader_val
                root_path[log_id] = root_path_val
            elif log_id in log_ids_test_og:
                loader[log_id] = loader_test
                root_path[log_id] = root_path_test

for log_id in log_ids_val:
            if log_id in log_ids_train_og:
                loader[log_id] = loader_train
                root_path[log_id] = root_path_train
            elif log_id in log_ids_val_og:
                loader[log_id] = loader_val
                root_path[log_id] = root_path_val
            elif log_id in log_ids_test_og:
                loader[log_id] = loader_test
                root_path[log_id] = root_path_test

for log_id in log_ids_test:
            if log_id in log_ids_train_og:
                loader[log_id] = loader_train
                root_path[log_id] = root_path_train
            elif log_id in log_ids_val_og:
                loader[log_id] = loader_val
                root_path[log_id] = root_path_val
            elif log_id in log_ids_test_og:
                loader[log_id] = loader_test
                root_path[log_id] = root_path_test

In [55]:
def get_osm_map_for_logids(log_ids, root_path):
    for log_id in log_ids:
        data_root = root_path[log_id]
        log_map_dirpath = Path(osp.join(data_root, log_id, "map"))
        vector_data_fnames = sorted(log_map_dirpath.glob("log_map_archive_*.json"))
        if not len(vector_data_fnames) == 1:
            raise RuntimeError(f"JSON file containing vector map data is missing (searched in {log_map_dirpath})")
        vector_data_fname = vector_data_fnames[0]
        vector_data_json_path = vector_data_fname

        avm = ArgoverseStaticMap.from_json(vector_data_json_path)
        av2_city_name = avm.log_id[-14:-11]
        scene_ls_list = avm.get_scenario_lane_segments()
        scene_ls_polygon_list = []
        for ls in scene_ls_list:
           scene_ls_polygon_list.append(Polygon(ls.polygon_boundary))

        if avm.log_id not in already_processed_scenario_maps:
            print("AV2 Map log_id: " + avm.log_id)
            bounds = MultiPolygon(scene_ls_polygon_list).bounds
            print("Scenario Map Bounds: " + str(bounds))
            bounds_with_buffer = list(deepcopy(bounds))
            bounds_with_buffer[0] -= 100
            bounds_with_buffer[1] -= 100
            bounds_with_buffer[2] += 100
            bounds_with_buffer[3] += 100
            print("Scenario Map Bounds with Buffer: " + str(bounds_with_buffer))

            bounds_with_buffer_latlon = convert_city_coords_to_wgs84(np.array([[bounds_with_buffer[0], bounds_with_buffer[1]], [bounds_with_buffer[2], bounds_with_buffer[3]]]), av2_city_name)

            print("Scenario Map Bounds with Buffer in Lat/Lon: " + str(bounds_with_buffer_latlon))

            query_str = "(node({0},{1},{2},{3}); rel(bn)->.x; way({0},{1},{2},{3}); node(w)->.x; rel(bw);); out meta;".format(bounds_with_buffer_latlon[0, 0], bounds_with_buffer_latlon[0, 1], bounds_with_buffer_latlon[1, 0], bounds_with_buffer_latlon[1, 1])
            data = {
              'data': query_str
            }
            response = requests.post('https://overpass-api.de/api/interpreter', headers=overpass_headers, data=data)
            Path(out_path).mkdir(parents=True, exist_ok=True)
            with open(osp.join(out_path, avm.log_id + '.osm'), 'w') as f:
                f.write(response.text)
            already_processed_scenario_maps[avm.log_id] = True
        else:
            #print("AV2 Map " + avm.log_id + " already processed, skipping...")
            pass

In [ ]:
get_osm_map_for_logids(log_ids_train, root_path)
get_osm_map_for_logids(log_ids_val, root_path)
get_osm_map_for_logids(log_ids_test, root_path)